# 01 - Pull raw SUUMO parser data

Mục tiêu của notebook này là tạo **data gốc trước** cho EDA. Dữ liệu gốc ở đây là JSON parser records do `suumo_page` tạo trong MinIO, được kéo về thành CSV local để các bước sau không phải chạm trực tiếp vào VPS.

Notebook này tự chứa toàn bộ code cần chạy: đọc env gần nhất (`notebook/.env` trước, nếu không có thì dùng root `.env.production`), query `load_batches`, tải file JSON gzip từ MinIO, đổi key tiếng Nhật sang tên cột tiếng Anh, và ghi raw CSV. Không cần chạy `pull_data.py` hay import `utils/*.py`.

## Bước 1 - Cấu hình output

Cell này chỉ khai báo đường dẫn. `suumo_parser_records_raw.csv` là cache raw dùng làm đầu vào cho notebook 02. File legacy `suumo_parser_records_eda.csv` chỉ được dùng để đọc lại dữ liệu cũ nếu bạn chưa refresh.

In [ ]:
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 100)

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != 'notebook':
    NOTEBOOK_DIR = NOTEBOOK_DIR / 'notebook'
DATA_DIR = NOTEBOOK_DIR / 'data'
RAW_PATH = DATA_DIR / 'suumo_parser_records_raw.csv'
LEGACY_RAW_PATH = DATA_DIR / 'suumo_parser_records_eda.csv'
RAW_PATH

## Bước 2 - Inline source readers

Các hàm dưới đây thay thế cho các file Python cũ. Chúng chỉ đọc source: PostgreSQL được mở bằng `default_transaction_read_only=on`, còn MinIO chỉ download object trong prefix `data/`. Env được resolve theo thứ tự `notebook/.env` rồi root `.env.production`.

In [ ]:


# ---- Environment settings ----
import os
from dataclasses import dataclass
from pathlib import Path

from dotenv import load_dotenv


NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "notebook":
    NOTEBOOK_DIR = NOTEBOOK_DIR / "notebook"
ROOT_DIR = NOTEBOOK_DIR.parent
DEFAULT_ENV_PATH = NOTEBOOK_DIR / ".env"
ROOT_PRODUCTION_ENV_PATH = ROOT_DIR / ".env.production"
ENV_CANDIDATE_PATHS = [DEFAULT_ENV_PATH, ROOT_PRODUCTION_ENV_PATH]


@dataclass(frozen=True)
class NotebookSettings:
    pg_host: str
    pg_port: int
    pg_db: str
    pg_user: str
    pg_password: str
    pg_connect_timeout: int
    minio_endpoint: str
    minio_access_key: str
    minio_secret_key: str
    minio_secure: bool
    minio_bucket: str
    minio_prefix: str


def _required(name: str) -> str:
    value = os.getenv(name, "").strip()
    if not value:
        raise ValueError(f"Missing required environment variable: {name}")
    return value


def _parse_bool(value: str) -> bool:
    normalized = value.strip().lower()
    if normalized not in {"true", "false"}:
        raise ValueError(f"Expected true or false, got {value!r}")
    return normalized == "true"


def resolve_env_path(env_path: Path | str | None = None) -> Path:
    if env_path is not None:
        resolved_path = Path(env_path)
        if not resolved_path.exists():
            raise FileNotFoundError(f"Missing env file: {resolved_path}")
        return resolved_path

    for candidate_path in ENV_CANDIDATE_PATHS:
        if candidate_path.exists():
            return candidate_path

    raise FileNotFoundError(
        "Missing notebook env. Create notebook/.env for notebook-local credentials "
        f"or {ROOT_PRODUCTION_ENV_PATH} for root-level production/VPS credentials."
    )


def load_settings(env_path: Path | str | None = None) -> NotebookSettings:
    env_path = resolve_env_path(env_path)
    print(f"Loading notebook connection settings from: {env_path}")
    load_dotenv(env_path, override=True)
    minio_endpoint = _required("NOTEBOOK_MINIO_ENDPOINT")
    if "://" in minio_endpoint:
        raise ValueError("NOTEBOOK_MINIO_ENDPOINT must be host:port, without http:// or https://")

    minio_prefix = os.getenv("NOTEBOOK_MINIO_PREFIX", "data/").strip("/") + "/"
    return NotebookSettings(
        pg_host=_required("NOTEBOOK_PG_HOST"),
        pg_port=int(os.getenv("NOTEBOOK_PG_PORT", "5432")),
        pg_db=_required("NOTEBOOK_PG_DB"),
        pg_user=_required("NOTEBOOK_PG_USER"),
        pg_password=_required("NOTEBOOK_PG_PASSWORD"),
        pg_connect_timeout=int(os.getenv("NOTEBOOK_PG_CONNECT_TIMEOUT", "10")),
        minio_endpoint=minio_endpoint,
        minio_access_key=_required("NOTEBOOK_MINIO_ACCESS_KEY"),
        minio_secret_key=_required("NOTEBOOK_MINIO_SECRET_KEY"),
        minio_secure=_parse_bool(os.getenv("NOTEBOOK_MINIO_SECURE", "false")),
        minio_bucket=os.getenv("NOTEBOOK_MINIO_BUCKET", "suumo").strip(),
        minio_prefix=minio_prefix,
    )


# ---- Source DB and MinIO readers ----
import gzip
import json
from dataclasses import dataclass
from datetime import datetime
from typing import Any, Callable

import psycopg
from minio import Minio


@dataclass(frozen=True)
class BatchMetadata:
    batch_id: int
    source_id: int
    file_path: str
    file_format: str
    compression: str | None
    row_count: int
    status: str
    inserted_count: int
    failed_count: int
    created_at: datetime
    started_loading_at: datetime | None
    finished_loading_at: datetime | None
    loaded_at: datetime | None


def postgres_connection(settings: NotebookSettings):
    return psycopg.connect(
        host=settings.pg_host,
        port=settings.pg_port,
        dbname=settings.pg_db,
        user=settings.pg_user,
        password=settings.pg_password,
        connect_timeout=settings.pg_connect_timeout,
        options="-c default_transaction_read_only=on",
    )


def minio_client(settings: NotebookSettings) -> Minio:
    return Minio(
        settings.minio_endpoint,
        access_key=settings.minio_access_key,
        secret_key=settings.minio_secret_key,
        secure=settings.minio_secure,
    )


def check_connections(settings: NotebookSettings) -> dict[str, str]:
    with postgres_connection(settings) as connection:
        with connection.cursor() as cursor:
            cursor.execute("SELECT current_database(), current_user")
            database, user = cursor.fetchone()

    client = minio_client(settings)
    if not client.bucket_exists(settings.minio_bucket):
        raise RuntimeError(f"MinIO bucket does not exist: {settings.minio_bucket}")

    return {
        "postgres": f"{settings.pg_host}:{settings.pg_port}/{database} as {user}",
        "minio": f"{'https' if settings.minio_secure else 'http'}://{settings.minio_endpoint}/{settings.minio_bucket}",
    }


def fetch_latest_batches(settings: NotebookSettings, limit: int = 180) -> list[BatchMetadata]:
    if limit <= 0:
        raise ValueError("limit must be greater than zero")

    with postgres_connection(settings) as connection:
        with connection.cursor() as cursor:
            cursor.execute(
                """
                SELECT
                    batch_id,
                    source_id,
                    file_path,
                    file_format,
                    compression,
                    row_count,
                    status::text,
                    inserted_count,
                    failed_count,
                    created_at,
                    started_loading_at,
                    finished_loading_at,
                    loaded_at
                FROM load_batches
                ORDER BY created_at DESC, batch_id DESC
                LIMIT %s
                """,
                (limit,),
            )
            rows = cursor.fetchall()

    return [BatchMetadata(*row) for row in rows]


def _object_name(settings: NotebookSettings, file_path: str) -> str:
    normalized = file_path.strip("/")
    bucket_prefix = f"{settings.minio_bucket}/"
    if normalized.startswith(bucket_prefix):
        normalized = normalized[len(bucket_prefix) :]
    if not normalized.startswith(settings.minio_prefix):
        raise ValueError(
            f"Batch file is outside MinIO prefix {settings.minio_prefix!r}: {file_path}"
        )
    return normalized


def _read_batch(client: Minio, settings: NotebookSettings, batch: BatchMetadata) -> list[dict[str, Any]]:
    if batch.file_format.lower() != "json":
        raise ValueError(f"Unsupported file format {batch.file_format!r} for batch {batch.batch_id}")

    object_name = _object_name(settings, batch.file_path)
    response = client.get_object(settings.minio_bucket, object_name)
    try:
        payload = response.read()
    finally:
        response.close()
        response.release_conn()

    if batch.compression == "gzip" or object_name.endswith(".gz"):
        payload = gzip.decompress(payload)
    elif batch.compression:
        raise ValueError(f"Unsupported compression {batch.compression!r} for batch {batch.batch_id}")

    records = json.loads(payload.decode("utf-8"))
    if not isinstance(records, list):
        raise ValueError(f"Batch {batch.batch_id} JSON payload must be an array")
    if not all(isinstance(record, dict) for record in records):
        raise ValueError(f"Batch {batch.batch_id} contains a non-object record")
    if len(records) != batch.row_count:
        raise ValueError(
            f"Batch {batch.batch_id} row count mismatch: metadata={batch.row_count}, JSON={len(records)}"
        )
    return records


def fetch_minio_records(
    settings: NotebookSettings,
    batches: list[BatchMetadata],
    progress: Callable[[int, int, BatchMetadata, int], None] | None = None,
) -> list[tuple[BatchMetadata, dict[str, Any]]]:
    client = minio_client(settings)
    fetched: list[tuple[BatchMetadata, dict[str, Any]]] = []
    total = len(batches)

    for index, batch in enumerate(batches, start=1):
        records = _read_batch(client, settings, batch)
        fetched.extend((batch, record) for record in records)
        if progress is not None:
            progress(index, total, batch, len(records))

    return fetched


# ---- Parser-record dataframe builder ----
import re
from typing import Any

import pandas as pd


JAPANESE_TO_ENGLISH = {
    "家賃": "rent_price_text",
    "敷金": "deposit_text",
    "管理費_共益費": "management_fee_text",
    "礼金": "key_money_text",
    "保証金": "guarantee_deposit_text",
    "敷引_償却": "depreciation_text",
    "電話番号": "phone_number",
    "所在地": "address",
    "駅徒歩": "station_access",
    "間取り": "layout",
    "専有面積": "exclusive_area_text",
    "築年数": "building_age_text",
    "階": "floor_text",
    "向き": "direction",
    "建物種別": "building_type",
    "間取り詳細": "layout_detail",
    "構造": "structure",
    "階建": "building_floors",
    "築年月": "built_at_text",
    "エネルギー消費性能": "energy_efficiency",
    "断熱性能": "insulation_performance",
    "目安光熱費": "estimated_utility_cost",
    "損保": "insurance",
    "駐車場": "parking",
    "入居": "move_in",
    "条件": "conditions",
    "SUUMO物件コード": "suumo_property_code",
    "情報更新日": "information_updated_at_text",
    "契約期間": "contract_period",
    "仲介手数料": "brokerage_fee",
    "保証会社": "guarantee_company",
    "ほか初期費用": "other_initial_costs",
    "ほか諸費用": "other_monthly_costs",
    "取引態様": "transaction_type",
    "取り扱い店舗物件コード": "shop_property_code",
    "総戸数": "total_units",
    "次回更新予定日": "next_update_date_text",
    "備考": "remarks",
}

_JAPANESE_PATTERN = re.compile(r"[぀-ヿ㐀-鿿]")

_BATCH_COLUMNS = [
    "batch_id",
    "source_id",
    "batch_file_path",
    "batch_status",
    "batch_row_count",
    "batch_inserted_count",
    "batch_failed_count",
    "batch_created_at",
    "batch_started_loading_at",
    "batch_finished_loading_at",
    "batch_loaded_at",
]


def contains_japanese(value: str) -> bool:
    return bool(_JAPANESE_PATTERN.search(value))


def _batch_values(batch: BatchMetadata) -> dict[str, Any]:
    return {
        "batch_id": batch.batch_id,
        "source_id": batch.source_id,
        "batch_file_path": batch.file_path,
        "batch_status": batch.status,
        "batch_row_count": batch.row_count,
        "batch_inserted_count": batch.inserted_count,
        "batch_failed_count": batch.failed_count,
        "batch_created_at": batch.created_at,
        "batch_started_loading_at": batch.started_loading_at,
        "batch_finished_loading_at": batch.finished_loading_at,
        "batch_loaded_at": batch.loaded_at,
    }


def build_dataframe(
    fetched_records: list[tuple[BatchMetadata, dict[str, Any]]],
) -> pd.DataFrame:
    unknown_japanese_keys = sorted(
        {
            key
            for _, record in fetched_records
            for key in record
            if contains_japanese(key) and key not in JAPANESE_TO_ENGLISH
        }
    )
    if unknown_japanese_keys:
        raise ValueError(
            "Missing Japanese-to-English mappings for keys: "
            + ", ".join(unknown_japanese_keys)
        )

    rows: list[dict[str, Any]] = []
    for batch, record in fetched_records:
        transformed = {
            JAPANESE_TO_ENGLISH.get(key, key): value for key, value in record.items()
        }
        collisions = set(_BATCH_COLUMNS).intersection(transformed)
        if collisions:
            raise ValueError(
                f"Parser record contains reserved batch metadata keys: {sorted(collisions)}"
            )
        rows.append({**_batch_values(batch), **transformed})

    dataframe = pd.DataFrame(rows)
    dataframe = dataframe.replace(r"^\s*$", pd.NA, regex=True)

    metadata_columns = [column for column in _BATCH_COLUMNS if column in dataframe.columns]
    record_columns = [column for column in dataframe.columns if column not in metadata_columns]
    return dataframe[metadata_columns + record_columns]


## Bước 3 - Refresh raw data từ source

Chạy cell này khi muốn lấy snapshot mới. `BATCH_LIMIT` nên giữ đủ lớn để có nhiều lịch sử cho EDA, nhưng không quá lớn nếu chỉ đang thử notebook.

In [ ]:
BATCH_LIMIT = 180

def progress(index, total, batch, record_count):
    print(f'[{index:>3}/{total}] batch_id={batch.batch_id} records={record_count} file={batch.file_path}')

def pull_raw_snapshot(limit=BATCH_LIMIT, output_path=RAW_PATH):
    settings = load_settings()
    connections = check_connections(settings)
    print(f'PostgreSQL read-only connection: {connections["postgres"]}')
    print(f'MinIO read-only connection: {connections["minio"]}')

    batches = fetch_latest_batches(settings, limit=limit)
    if not batches:
        raise RuntimeError('No load_batches rows were found')
    print(f'Found {len(batches)} latest batches')

    fetched_records = fetch_minio_records(settings, batches, progress=progress)
    raw = build_dataframe(fetched_records)

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    raw.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f'Saved raw parser records: {len(raw):,} rows x {len(raw.columns):,} columns')
    print(f'CSV: {output_path.resolve()}')
    return raw, batches

# Uncomment this line when you need a fresh raw snapshot from PostgreSQL + MinIO.
# raw_df, batches = pull_raw_snapshot(limit=BATCH_LIMIT, output_path=RAW_PATH)

## Bước 4 - Load raw cache local

Cell này là đường chạy thường ngày. Nếu đã có raw CSV, notebook 02 có thể chạy offline hoàn toàn.

In [ ]:
cache_path = RAW_PATH if RAW_PATH.exists() else LEGACY_RAW_PATH
if not cache_path.exists():
    raise FileNotFoundError('Chưa có raw cache. Điền notebook/.env rồi chạy pull_raw_snapshot ở bước 3.')

raw_df = pd.read_csv(
    cache_path,
    encoding='utf-8-sig',
    low_memory=False,
    dtype={
        'suumo_property_code': 'string',
        'shop_property_code': 'string',
        'phone_number': 'string',
        'data_hash': 'string',
    },
)
if cache_path != RAW_PATH:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    raw_df.to_csv(RAW_PATH, index=False, encoding='utf-8-sig')
    print(f'Copied legacy raw cache to {RAW_PATH}')

print(f'Rows: {len(raw_df):,}')
print(f'Columns: {len(raw_df.columns):,}')
print(f'Batches: {raw_df["batch_id"].nunique():,}')
print(f'Raw cache: {RAW_PATH.resolve()}')

## Bước 5 - Kiểm tra nhanh raw data

Không hiển thị địa chỉ, số điện thoại, ghi chú, hoặc free-text nhạy cảm. Mục đích là xác nhận raw snapshot có đủ record, batch, và cột nguồn trước khi qua notebook 02.

In [ ]:
safe_preview_columns = [
    'task_id', 'batch_id', 'suumo_property_code', 'rent_price_text',
    'management_fee_text', 'layout', 'exclusive_area_text',
    'building_age_text', 'floor_text', 'direction', 'building_type',
]
raw_df[safe_preview_columns].head(10)

In [ ]:
raw_overview = pd.DataFrame({
    'dtype': raw_df.dtypes.astype(str),
    'non_null': raw_df.notna().sum(),
    'missing': raw_df.isna().sum(),
    'missing_pct': (raw_df.isna().mean() * 100).round(2),
    'unique_non_null': raw_df.nunique(dropna=True),
}).sort_values(['missing_pct', 'unique_non_null'], ascending=[False, True])
raw_overview

## Kết quả

Sau notebook này, đầu vào chính cho EDA/cleaning là `notebook/data/suumo_parser_records_raw.csv`. Chạy tiếp `02_column_eda.ipynb`; notebook đó sẽ phân tích từng column/value trước rồi mới tạo các CSV clean.